# Experiment 07 — Linking It All Together: A Small Conversational Agent

```
Language comprehension
      ↓
Meaning + memory + emotion + social context
      ↓
Response planning
      ↓
Word and sentence construction
```

Experiments 02-05 each built one stage of this pipeline as an island: every
notebook synthesized stand-in inputs for what the previous stage would have
produced, rather than actually consuming it. This notebook does the wiring
for real — one `ConversationalAgent` module made of the same four pieces,
tensors flowing genuinely from stage to stage, plus one thing none of the
standalone notebooks had: a **persistent memory** that carries across turns
within a conversation, so the agent can hold an actual multi-turn exchange
instead of processing each sentence in isolation.

(Stage 06, motor commands, stays unconnected — turning generated words into
articulation doesn't change how the conversation *behaves*, so it's left out
of this pipeline on purpose, not forgotten.)

We train it on a small set of hand-written toy conversations and then
actually test whether the wiring works: does memory change what the agent
does? Does it produce anything sensible on a conversation it never saw?

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0);

## A small set of toy conversations

Same label categories as experiments 02-05 (intent, emotion, tone, plan), but
now attached to real multi-turn conversations instead of independent
sentences. Each turn has: the user's utterance, four ground-truth labels for
the auxiliary classifiers, three social features, and the target response.

One turn is special on purpose: **"okay" appears twice**, in two different
conversations, with *identical* text, emotion label, and social features —
the only thing that differs is what was said before it. If the agent is
actually using memory (not just reacting to the current sentence), it should
give these two identical-looking turns different plans and different
responses. If it isn't, it can't — this is the cleanest test we can build for
"is memory actually doing anything."

In [2]:
intents = ["question", "statement", "greeting", "command"]
emotions = ["neutral", "happy", "sad", "angry", "anxious"]
tones = ["supportive", "formal", "playful", "urgent"]
plans = ["answer_directly", "ask_clarifying_question", "empathize", "give_instruction"]

train_conversations = [
    [   # A: friendly small talk
        dict(user="hello there friend", intent="greeting", emotion="happy", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="hello it is good to see you"),
        dict(user="how are you today", intent="question", emotion="happy", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="i am doing well thank you"),
        dict(user="nice to meet you", intent="greeting", emotion="happy", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="nice to meet you too"),
        dict(user="okay", intent="statement", emotion="neutral", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="great let us continue"),
    ],
    [   # B: formal question/answer
        dict(user="what time is the meeting", intent="question", emotion="neutral", formality=0.8, closeness=0.3, urgency=0.3,
             tone="formal", plan="answer_directly", response="the meeting starts at three"),
        dict(user="where is the file", intent="question", emotion="neutral", formality=0.8, closeness=0.3, urgency=0.3,
             tone="formal", plan="ask_clarifying_question", response="which file do you mean"),
    ],
    [   # C: distress -> empathize (the "okay" contrast case lives here)
        dict(user="i am really stressed about this", intent="statement", emotion="anxious", formality=0.3, closeness=0.7, urgency=0.5,
             tone="supportive", plan="empathize", response="that sounds really hard"),
        dict(user="i do not know what to do", intent="statement", emotion="sad", formality=0.3, closeness=0.7, urgency=0.5,
             tone="supportive", plan="empathize", response="i hear you and that matters"),
        dict(user="okay", intent="statement", emotion="neutral", formality=0.2, closeness=0.6, urgency=0.1,
             tone="supportive", plan="empathize", response="take your time i am here for you"),
        dict(user="thank you for listening", intent="statement", emotion="happy", formality=0.3, closeness=0.7, urgency=0.2,
             tone="playful", plan="answer_directly", response="i am glad i could help"),
    ],
    [   # D: commands
        dict(user="please close the door", intent="command", emotion="neutral", formality=0.6, closeness=0.4, urgency=0.4,
             tone="formal", plan="give_instruction", response="closing the door now"),
        dict(user="turn off the lights", intent="command", emotion="neutral", formality=0.6, closeness=0.4, urgency=0.4,
             tone="formal", plan="give_instruction", response="turning off the lights"),
    ],
    [   # E: ambiguous -> clarify -> instruct
        dict(user="can you fix it", intent="question", emotion="neutral", formality=0.4, closeness=0.5, urgency=0.5,
             tone="urgent", plan="ask_clarifying_question", response="which one do you mean"),
        dict(user="the printer upstairs", intent="statement", emotion="neutral", formality=0.4, closeness=0.5, urgency=0.6,
             tone="urgent", plan="give_instruction", response="restart the device now"),
    ],
]

# held out entirely -- never used for training, only as a qualitative probe later
probe_conversation = [
    dict(user="this is not working at all", formality=0.3, closeness=0.4, urgency=0.7),
    dict(user="still broken", formality=0.3, closeness=0.4, urgency=0.8),
]

vocab = {"<bos>": 0, "<eos>": 1, "<unk>": 2}
for conv in train_conversations:
    for turn in conv:
        for w in (turn["user"] + " " + turn["response"]).split():
            vocab.setdefault(w, len(vocab))
id_to_word = {i: w for w, i in vocab.items()}

n_turns = sum(len(c) for c in train_conversations)
print(f"{len(train_conversations)} conversations, {n_turns} training turns, vocab size {len(vocab)}")


def encode_words(sentence):
    return [vocab.get(w, vocab["<unk>"]) for w in sentence.split()]


def encode_response(sentence):
    return [vocab["<bos>"]] + encode_words(sentence) + [vocab["<eos>"]]

5 conversations, 14 training turns, vocab size 76


## Architecture: four stages, one memory, real tensors between them

Each stage is close to its standalone notebook, with two real changes now that
they're actually connected:

- **Comprehension** now also predicts **emotion** from the sentence (a new
  head on the same hidden vector) instead of receiving it as a hand-fed
  input — a real conversational partner isn't told the other person's
  emotion, it has to infer it.
- **Generation** now conditions on `[fused_vector ; plan_embedding]`, not
  just the plan embedding alone. Stage 05 conditioned only on plan (4
  categories total), so every turn sharing a plan was *forced* to collapse
  onto one memorized sentence — there was no way to say two different things
  under the same plan. Concatenating in the fused vector (which is
  turn-specific, carrying the actual meaning + memory of this turn) fixes
  that: two "give_instruction" turns can now produce different sentences,
  because they differ in more than just their plan label.
- A **memory `GRUCell`** sits between turns: after each turn, this turn's
  comprehension vector updates a running memory state, which becomes the
  "memory" input to fusion on the *next* turn. Reset to zero at the start of
  each new conversation.

**Training vs. inference differ in one way:** discrete choices (which
emotion, which plan) use the ground-truth label during training
(teacher forcing, so early mistakes in one classifier don't cascade and
corrupt training of everything downstream) but the model's own prediction
during inference (fully autonomous — no ground truth available for a real
conversation).

In [3]:
class Comprehension(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=16, n_intents=4, n_emotions=5):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.to_comprehension = nn.Linear(embed_dim, hidden_dim)
        self.intent_head = nn.Linear(hidden_dim, n_intents)
        self.emotion_head = nn.Linear(hidden_dim, n_emotions)

    def forward(self, token_ids):
        pooled = self.embed(token_ids).mean(dim=0)
        comprehension = torch.tanh(self.to_comprehension(pooled))
        return comprehension, self.intent_head(comprehension), self.emotion_head(comprehension)


class Fusion(nn.Module):
    def __init__(self, meaning_dim=16, memory_dim=16, n_emotions=5, social_dim=3, fused_dim=16, n_tones=4):
        super().__init__()
        self.meaning_proj = nn.Linear(meaning_dim, fused_dim)
        self.memory_proj = nn.Linear(memory_dim, fused_dim)
        self.emotion_embed = nn.Embedding(n_emotions, fused_dim)
        self.social_proj = nn.Linear(social_dim, fused_dim)
        self.tone_head = nn.Sequential(nn.Linear(fused_dim, fused_dim), nn.ReLU(), nn.Linear(fused_dim, n_tones))

    def forward(self, meaning, memory, emotion_idx, social):
        fused = torch.tanh(
            self.meaning_proj(meaning) + self.memory_proj(memory)
            + self.emotion_embed(emotion_idx) + self.social_proj(social)
        )
        return fused, self.tone_head(fused)


class Planner(nn.Module):
    def __init__(self, fused_dim=16, hidden_dim=32, n_plans=4, plan_dim=8):
        super().__init__()
        self.classifier = nn.Sequential(nn.Linear(fused_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, n_plans))
        self.plan_embed = nn.Embedding(n_plans, plan_dim)

    def forward(self, fused):
        return self.classifier(fused)

    def embed(self, plan_idx):
        return self.plan_embed(plan_idx)


class Generator(nn.Module):
    def __init__(self, vocab_size, context_dim, word_dim=16, hidden_dim=32):
        super().__init__()
        self.word_embed = nn.Embedding(vocab_size, word_dim)
        self.cell = nn.GRUCell(word_dim + context_dim, hidden_dim)
        self.to_vocab = nn.Linear(hidden_dim, vocab_size)
        self.hidden_dim = hidden_dim

    def forward_sequence(self, token_ids, context):
        h = torch.zeros(1, self.hidden_dim)
        context = context.unsqueeze(0)
        logits_list = []
        for t in range(len(token_ids) - 1):
            w = self.word_embed(torch.tensor(token_ids[t])).unsqueeze(0)
            h = self.cell(torch.cat([w, context], dim=1), h)
            logits_list.append(self.to_vocab(h).squeeze(0))
        return torch.stack(logits_list)

    def generate(self, context, max_len=10):
        h = torch.zeros(1, self.hidden_dim)
        context = context.unsqueeze(0)
        token = vocab["<bos>"]
        out = []
        for _ in range(max_len):
            w = self.word_embed(torch.tensor(token)).unsqueeze(0)
            h = self.cell(torch.cat([w, context], dim=1), h)
            token = self.to_vocab(h).squeeze(0).argmax().item()
            if token == vocab["<eos>"]:
                break
            out.append(token)
        return out

## The agent: wires the four stages together through one memory state

In [4]:
class ConversationalAgent(nn.Module):
    def __init__(self, vocab_size, comprehension_dim=16, memory_dim=16, fused_dim=16, plan_dim=8):
        super().__init__()
        self.comprehension = Comprehension(vocab_size, hidden_dim=comprehension_dim)
        self.memory_cell = nn.GRUCell(comprehension_dim, memory_dim)
        self.fusion = Fusion(meaning_dim=comprehension_dim, memory_dim=memory_dim, fused_dim=fused_dim)
        self.planner = Planner(fused_dim=fused_dim, plan_dim=plan_dim)
        self.generator = Generator(vocab_size, context_dim=fused_dim + plan_dim)
        self.memory_dim = memory_dim

    def step(self, turn, memory, teacher_force):
        """Process one conversational turn. teacher_force=True uses ground-truth
        emotion/plan labels (training); False uses the model's own predictions
        and autoregressively generates a response (inference)."""
        user_ids = torch.tensor(encode_words(turn["user"]))
        comprehension, intent_logits, emotion_logits = self.comprehension(user_ids)

        emotion_idx = torch.tensor(emotions.index(turn["emotion"])) if teacher_force else emotion_logits.argmax()
        social = torch.tensor([turn["formality"], turn["closeness"], turn["urgency"]], dtype=torch.float32)
        fused, tone_logits = self.fusion(comprehension, memory, emotion_idx, social)

        plan_logits = self.planner(fused)
        plan_idx = torch.tensor(plans.index(turn["plan"])) if teacher_force else plan_logits.argmax()
        context = torch.cat([fused, self.planner.embed(plan_idx)])

        result = dict(
            predicted_intent=intents[intent_logits.argmax().item()],
            predicted_emotion=emotions[emotion_logits.argmax().item()],
            predicted_tone=tones[tone_logits.argmax().item()],
            predicted_plan=plans[plan_logits.argmax().item()],
        )

        if teacher_force:
            response_ids = encode_response(turn["response"])
            gen_logits = self.generator.forward_sequence(response_ids, context)
            result["loss"] = (
                F.cross_entropy(intent_logits.unsqueeze(0), torch.tensor([intents.index(turn["intent"])]))
                + F.cross_entropy(emotion_logits.unsqueeze(0), torch.tensor([emotions.index(turn["emotion"])]))
                + F.cross_entropy(tone_logits.unsqueeze(0), torch.tensor([tones.index(turn["tone"])]))
                + F.cross_entropy(plan_logits.unsqueeze(0), torch.tensor([plans.index(turn["plan"])]))
                + F.cross_entropy(gen_logits, torch.tensor(response_ids[1:]))
            )
        else:
            result["response_text"] = " ".join(id_to_word[t] for t in self.generator.generate(context))

        new_memory = self.memory_cell(comprehension.unsqueeze(0), memory.unsqueeze(0)).squeeze(0)
        return new_memory, result


agent = ConversationalAgent(vocab_size=len(vocab))
n_params = sum(p.numel() for p in agent.parameters())
print(f"agent parameter count: {n_params:,}")

agent parameter count: 15,837


## Training

Sum all five losses (intent, emotion, tone, plan, generation) over every turn
of every conversation, memory carried within a conversation and reset between
conversations, one optimizer step per epoch — same pattern as stage 05's
per-sentence loop, just with a persistent memory state and more supervision
signals stacked on top.

In [5]:
optimizer = torch.optim.Adam(agent.parameters(), lr=0.01)

losses = []
for epoch in range(500):
    total_loss = 0.0
    optimizer.zero_grad()
    for conv in train_conversations:
        memory = torch.zeros(agent.memory_dim)
        for turn in conv:
            memory, result = agent.step(turn, memory, teacher_force=True)
            total_loss = total_loss + result["loss"]
    total_loss.backward()
    optimizer.step()
    losses.append(total_loss.item())

print(f"total loss: {losses[0]:.2f} -> {losses[-1]:.4f}")

with torch.no_grad():
    correct = dict(intent=0, emotion=0, tone=0, plan=0)
    for conv in train_conversations:
        memory = torch.zeros(agent.memory_dim)
        for turn in conv:
            memory, result = agent.step(turn, memory, teacher_force=True)
            correct["intent"] += result["predicted_intent"] == turn["intent"]
            correct["emotion"] += result["predicted_emotion"] == turn["emotion"]
            correct["tone"] += result["predicted_tone"] == turn["tone"]
            correct["plan"] += result["predicted_plan"] == turn["plan"]
    for k, v in correct.items():
        print(f"{k} train accuracy: {v}/{n_turns} = {v / n_turns:.2%}")

total loss: 143.25 -> 0.0389
intent train accuracy: 14/14 = 100.00%
emotion train accuracy: 14/14 = 100.00%
tone train accuracy: 14/14 = 100.00%
plan train accuracy: 14/14 = 100.00%


## Does the wiring actually produce a conversation?

Run every training conversation through the agent again, this time
autoregressively generating each response from `<bos>` (still using the
ground-truth plan/emotion for context, so we're checking "can it say the
right thing," not yet "can it choose the right thing" — that's the next
section).

In [6]:
with torch.no_grad():
    for conv in train_conversations:
        memory = torch.zeros(agent.memory_dim)
        for turn in conv:
            user_ids = torch.tensor(encode_words(turn["user"]))
            comprehension, _, _ = agent.comprehension(user_ids)
            emotion_idx = torch.tensor(emotions.index(turn["emotion"]))
            social = torch.tensor([turn["formality"], turn["closeness"], turn["urgency"]], dtype=torch.float32)
            fused, _ = agent.fusion(comprehension, memory, emotion_idx, social)
            plan_vec = agent.planner.embed(torch.tensor(plans.index(turn["plan"])))
            context = torch.cat([fused, plan_vec])
            generated = " ".join(id_to_word[t] for t in agent.generator.generate(context))
            match = "match" if generated == turn["response"] else "DIFFERS"
            print(f"{turn['user']!r:35} -> {generated!r:38} [{match}]")
            memory = agent.memory_cell(comprehension.unsqueeze(0), memory.unsqueeze(0)).squeeze(0)

'hello there friend'                -> 'hello it is good to see you'          [match]
'how are you today'                 -> 'i am doing well thank you'            [match]
'nice to meet you'                  -> 'nice to meet you too'                 [match]
'okay'                              -> 'great let us continue'                [match]
'what time is the meeting'          -> 'the meeting starts at three'          [match]
'where is the file'                 -> 'which file do you mean'               [match]
'i am really stressed about this'   -> 'that sounds really hard'              [match]
'i do not know what to do'          -> 'i hear you and that matters'          [match]
'okay'                              -> 'take your time i am here for you'     [match]
'thank you for listening'           -> 'i am glad i could help'               [match]
'please close the door'             -> 'closing the door now'                 [match]
'turn off the lights'               -> 'turning off th

## Does memory actually change anything?

The real test: the two **"okay"** turns. Identical text, identical predicted
emotion, identical social features — the only difference between them is
what happened earlier in their respective conversations. We'll take the
"okay" from the distress conversation and run it two ways: with memory
genuinely carried from that conversation's first two turns, and with memory
forcibly zeroed out (as if this were the very first thing said). Everything
else about the input is held exactly fixed. If the agent's choice changes
between these two runs, that's not a training-data pattern-match — it's the
memory pathway doing real causal work, isolated from every other variable.

In [7]:
conv_c = train_conversations[2]
turn1, turn2, okay_turn = conv_c[0], conv_c[1], conv_c[2]

with torch.no_grad():
    memory_after_t1, _ = agent.step(turn1, torch.zeros(agent.memory_dim), teacher_force=True)
    memory_after_t2, _ = agent.step(turn2, memory_after_t1, teacher_force=True)

    _, result_carried = agent.step(okay_turn, memory_after_t2, teacher_force=False)
    _, result_reset = agent.step(okay_turn, torch.zeros(agent.memory_dim), teacher_force=False)

print(f"turn: {okay_turn['user']!r} (identical text, emotion, and social features in both runs below)")
print()
print("memory carried (real distress context):")
print(f"  plan={result_carried['predicted_plan']:24} tone={result_carried['predicted_tone']:10} response={result_carried['response_text']!r}")
print("memory reset (as if the prior turns never happened):")
print(f"  plan={result_reset['predicted_plan']:24} tone={result_reset['predicted_tone']:10} response={result_reset['response_text']!r}")

turn: 'okay' (identical text, emotion, and social features in both runs below)

memory carried (real distress context):
  plan=empathize                tone=supportive response='take your time i am here for you'
memory reset (as if the prior turns never happened):
  plan=empathize                tone=urgent     response='take your time i am here for you'


## A conversation it never saw

Two lines held out entirely — never used in any loss computation, never in
the vocabulary-building pass either, so some words will genuinely be
`<unk>`. Fully autonomous: the agent picks its own emotion and plan at every
turn, memory carried across the two lines like a real (very short)
conversation.

In [8]:
with torch.no_grad():
    memory = torch.zeros(agent.memory_dim)
    for turn in probe_conversation:
        memory, result = agent.step(turn, memory, teacher_force=False)
        print(f"{turn['user']!r:30} -> intent={result['predicted_intent']:10} emotion={result['predicted_emotion']:8} "
              f"tone={result['predicted_tone']:10} plan={result['predicted_plan']:24} response={result['response_text']!r}")

'this is not working at all'   -> intent=statement  emotion=neutral  tone=urgent     plan=ask_clarifying_question  response='which one do you mean'
'still broken'                 -> intent=statement  emotion=neutral  tone=urgent     plan=give_instruction         response='restart the device now'


## What actually happened

**The wiring works, end to end.** All four auxiliary classifiers (intent,
emotion, tone, plan) hit 100% train accuracy (14/14) on the real, connected
pipeline — not a stand-in — and all 14 generated responses matched their
targets exactly, including both "give_instruction" turns that share a plan
but need different words ("closing the door now" vs. "turning off the
lights") — the fused-vector conditioning fix over stage 05 is doing real
work, not just adding harmless extra input.

**Memory has a real, if partial, effect.** The two identical "okay" turns:
with memory genuinely carried from the distress conversation, predicted
`tone=supportive`; with memory forcibly zeroed (same text, same predicted
emotion, same social features), predicted `tone=urgent`. That flip is
memory's effect, isolated from every other variable — the plan classifier
still called it `empathize` either way, and since the generator conditions
on `[fused ; plan_embed]` and not on `tone` directly, the generated response
text happened to stay identical in this case even though the internal
`fused` vector (which does depend on memory) genuinely changed. So: memory
demonstrably changes the agent's read of the situation, but with a
14-turn dataset and only a few genuinely context-dependent examples, that
change doesn't always cascade all the way to different words on the page.

**The untrained probe is genuinely encouraging.** "this is not working at
all" / "still broken" — 4 of its 8 distinct words ("working," "all,"
"still," "broken") never appeared in training — got `ask_clarifying_question`
-> "which one do you mean" then `give_instruction` -> "restart the device
now": plausible, coherent, on-topic responses to input the agent never saw,
not nonsense. Whether that's real compositional generalization or this
particular pair of sentences happening to land close to conversation E in
the model's learned space is genuinely unclear at this scale — 14 training
turns isn't enough to tell those apart with confidence, and it would be
overclaiming to say more than "it didn't fall apart."

**What's still not wired in:** stage 06 (motor commands) — this agent
produces text, not articulation. **What's still purely toy:** 5
conversations and 14 turns is enough to prove the mechanism works, nowhere
near enough to hold a conversation about anything not in this notebook.
Scaling this up — more conversations, more genuinely ambiguous
memory-dependent cases like the "okay" pair, maybe eventually real dialogue
data — is the natural next expansion, not a rewrite of the architecture.

## Try it yourself

The agent only knows the ~73 real words it was trained on (everything else
becomes `<unk>`, same honest limitation as experiment 02) — so inputs built
from this vocabulary will get a real response; anything else will mostly
collapse to `<unk>` and produce something closer to noise. Print the
vocabulary first so you know what it can actually understand.

In [9]:
known_words = sorted(w for w in vocab if not w.startswith("<"))
print(f"{len(known_words)} known words:\n")
print(", ".join(known_words))

73 known words:

about, am, and, are, at, can, close, closing, continue, could, device, do, doing, door, file, fix, for, friend, glad, good, great, hard, hear, hello, help, here, how, i, is, it, know, let, lights, listening, matters, mean, meet, meeting, nice, not, now, off, okay, one, please, printer, really, restart, see, sounds, starts, stressed, take, thank, that, the, there, this, three, time, to, today, too, turn, turning, upstairs, us, well, what, where, which, you, your


In [10]:
def chat(text, memory, formality=0.5, closeness=0.5, urgency=0.5, verbose=True):
    """Run one turn through the trained agent, fully autonomous (no ground-truth
    labels). Returns (result_dict, new_memory) -- pass new_memory into the next
    call to keep the conversation going, or start a fresh torch.zeros(agent.memory_dim)
    to reset it."""
    turn = dict(user=text, formality=formality, closeness=closeness, urgency=urgency)
    with torch.no_grad():
        new_memory, result = agent.step(turn, memory, teacher_force=False)
    if verbose:
        print(f"you:   {text}")
        print(f"agent: {result['response_text']}  "
              f"[emotion={result['predicted_emotion']}, tone={result['predicted_tone']}, plan={result['predicted_plan']}]")
    return result, new_memory

A quick example, chaining memory across turns manually (a template for
writing your own — this exact 2-turn conversation was never in training,
though every individual word was):

In [11]:
memory = torch.zeros(agent.memory_dim)
_, memory = chat("can you turn off the meeting lights", memory)
_, memory = chat("thank you for that", memory)

# start a new, unrelated exchange by resetting memory back to zero:
# memory = torch.zeros(agent.memory_dim)

you:   can you turn off the meeting lights
agent: closing the door now  [emotion=neutral, tone=formal, plan=give_instruction]
you:   thank you for that
agent: i am glad i could help  [emotion=happy, tone=playful, plan=answer_directly]


## Live chat

Run the cell below directly in your own Jupyter/VS Code kernel — it's a real
`input()` loop, so it needs your keyboard, which means it can't be run
automatically as part of building this notebook (there's nothing to type
into). Type `reset` to start a new conversation (clears memory), `quit` to
stop.

If it just spins with no visible prompt: `input()` support in VS Code's
notebook UI is finicky — when it works, the prompt appears in a thin box at
the very top of the editor window, easy to miss; in some kernel/extension
setups it never appears at all. Click the cell's **stop** button to
interrupt it, then use the reliable fallback method just below instead.

In [ ]:
memory = torch.zeros(agent.memory_dim)
print("Chatting with the agent. Commands: 'reset' clears memory, 'quit' exits.\n")

while True:
    text = input("you: ").strip()
    if text.lower() == "quit":
        break
    if text.lower() == "reset":
        memory = torch.zeros(agent.memory_dim)
        print("(memory reset)\n")
        continue
    if not text:
        continue
    _, memory = chat(text, memory)
    print()

## Live chat, the reliable fallback

No `input()` needed. Run the cell below once to (re)start a conversation.
Then edit the message in the **next** cell and re-run just that cell
(click it, `Shift+Enter`) as many times as you like — `memory` is an
ordinary notebook variable, so it persists between runs and each rerun
becomes the next turn in the same conversation. Come back and rerun this
first cell whenever you want to start a fresh conversation.

In [12]:
# run this once to start (or restart) a conversation
memory = torch.zeros(agent.memory_dim)
print("conversation reset")

conversation reset


In [13]:
# edit the message below, then re-run just this cell to send it and see the reply
_, memory = chat("your message here", memory)

you:   your message here
agent: restart the device now  [emotion=neutral, tone=urgent, plan=give_instruction]
